In [1]:
# ==============================================================================
# === SUPER-AUDIT v13-ULTRA-FAST : MATRICE 4×5 (100% BITWISE EQUIVALENT) ===
# 🚀 Optimisations HFT avec garantie de fidélité mathématique totale
#
# ★ Slicing corrigé pour équivalence exacte avec l'ancien range()
# ★ Filets try/except restaurés sur tous les estimateurs
# ★ Switch de Backend Loky/Threads facile d'accès
# ==============================================================================

import numpy as np
import pandas as pd
import urllib.request, zipfile, io, warnings, sys, subprocess, time, os
from scipy.stats import mannwhitneyu
from scipy.signal import periodogram
from sklearn.decomposition import PCA
from numpy.lib.stride_tricks import sliding_window_view, as_strided
from concurrent.futures import ThreadPoolExecutor, as_completed
from joblib import Parallel, delayed
warnings.filterwarnings('ignore')

for lib, imp in [('scikit-dimension', 'skdim'), ('antropy', 'antropy'),
                 ('nolds', 'nolds'), ('statsmodels', 'statsmodels')]:
    try: __import__(imp)
    except ImportError: subprocess.check_call([sys.executable, '-m', 'pip', 'install', lib, '-q'])

import skdim, antropy, nolds
from statsmodels.stats.multitest import multipletests

# ==============================================================================
# ★ PARAMÈTRES DE PERFORMANCE
# ==============================================================================
STRIDE     = 12
WINDOW     = 120
N_JOBS     = -1         # -1 = tous CPUs
BACKEND    = "loky"     # Mettez "threads" si vous manquez de RAM ou si Colab plante
SKIP_DANKO = False

# ==============================================================================
# ⚙️ MOTEUR MATHÉMATIQUE MUTUALISÉ
# ==============================================================================
class WindowContext:
    __slots__ = ('x', 'xz', 'X3', 'X10')

    def __init__(self, x):
        self.x = x
        x_clip = np.clip(x, np.percentile(x, 1), np.percentile(x, 99))
        self.xz = (x_clip - np.mean(x_clip)) / (np.std(x_clip) + 1e-8)

        n3 = len(self.xz) - 2
        if n3 >= 10:
            self.X3 = as_strided(self.xz, shape=(n3, 3), strides=(self.xz.strides[0], self.xz.strides[0])).copy()
        else:
            self.X3 = None

        n10 = len(self.xz) - 9
        if n10 >= 10:
            self.X10 = as_strided(self.xz, shape=(n10, 10), strides=(self.xz.strides[0], self.xz.strides[0])).copy()
        else:
            self.X10 = None

# ==============================================================================
# ⚙️ 27 ESTIMATEURS (100% blindés avec try/except locaux)
# ==============================================================================
def E01_Hurst(ctx):
    try: return float(nolds.hurst_rs(ctx.xz))
    except: return np.nan
def E02_SampleEn(ctx):
    try: return float(antropy.sample_entropy(ctx.xz[:500], order=2))
    except: return np.nan
def E03_Lyapunov(ctx):
    try: return float(nolds.lyap_r(ctx.xz))
    except: return np.nan
def E04_DFA(ctx):
    try: return float(nolds.dfa(ctx.xz))
    except: return np.nan
def E05_PermEn(ctx):
    try: return float(antropy.perm_entropy(ctx.xz, order=5, normalize=True))
    except: return np.nan
def E06_SpectralEn(ctx):
    try:
        _, psd = periodogram(ctx.xz, fs=1.0)
        psd = psd[1:]; psd_n = psd / psd.sum()
        return float(-np.sum(psd_n * np.log(psd_n + 1e-12)))
    except: return np.nan
def E07_SVDEn(ctx):
    try: return float(antropy.svd_entropy(ctx.xz, order=3, delay=1, normalize=True))
    except: return np.nan
def E08_AppEn(ctx):
    try: return float(antropy.app_entropy(ctx.xz[:500], order=2))
    except: return np.nan
def E09_Katz(ctx):
    try: return float(antropy.katz_fd(ctx.xz))
    except: return np.nan
def E10_Higuchi(ctx):
    try: return float(antropy.higuchi_fd(ctx.xz))
    except: return np.nan
def E11_Sevcik(ctx):
    try:
        x_z = ctx.xz; n = len(x_z)
        xn = (x_z - x_z.min()) / (x_z.max() - x_z.min() + 1e-12)
        L = np.sum(np.sqrt(np.diff(xn)**2 + np.diff(np.linspace(0, 1, n))**2))
        return float(1 + np.log(L) / np.log(2 * (n - 1)))
    except: return np.nan
def E12_BoxCount(ctx):
    try:
        x_z = ctx.xz
        xn = (x_z - x_z.min()) / (x_z.max() - x_z.min() + 1e-12)
        scales = [4, 8, 16, 32, 64]
        counts = [np.sum(np.histogram(xn, bins=np.linspace(0, 1, s+1))[0] > 0) for s in scales]
        return float(-np.polyfit(np.log(scales), np.log(counts), 1)[0])
    except: return np.nan
def E13_CorrDim(ctx):
    try: return float(nolds.corr_dim(ctx.xz, emb_dim=3))
    except: return np.nan
def E14_LempelZiv(ctx):
    try:
        s = "".join((ctx.xz > np.median(ctx.xz)).astype(str))
        return float(antropy.lziv_complexity(s, normalize=True))
    except: return np.nan
def E15_PCA_dim(ctx):
    try:
        if ctx.X10 is None: return np.nan
        pca = PCA().fit(ctx.X10)
        return float(np.searchsorted(np.cumsum(pca.explained_variance_ratio_), 0.95) + 1)
    except: return np.nan

def E16_MADA(ctx):
    try: return float(skdim.id.MADA().fit(ctx.X3).dimension_) if ctx.X3 is not None else np.nan
    except: return np.nan
def E17_MiND_ML(ctx):
    try: return float(skdim.id.MiND_ML().fit(ctx.X3).dimension_) if ctx.X3 is not None else np.nan
    except: return np.nan
def E18_TwoNN(ctx):
    try: return float(skdim.id.TwoNN().fit(ctx.X3).dimension_) if ctx.X3 is not None else np.nan
    except: return np.nan
def E19_MLE(ctx):
    try: return float(skdim.id.MLE().fit(ctx.X3).dimension_) if ctx.X3 is not None else np.nan
    except: return np.nan
def E20_DANCo(ctx):
    try: return float(skdim.id.DANCo().fit(ctx.X3).dimension_) if ctx.X3 is not None else np.nan
    except: return np.nan
def E21_MOM(ctx):
    try: return float(skdim.id.MOM().fit(ctx.X3).dimension_) if ctx.X3 is not None else np.nan
    except: return np.nan
def E22_TLE(ctx):
    try: return float(skdim.id.TLE().fit(ctx.X3).dimension_) if ctx.X3 is not None else np.nan
    except: return np.nan
def E23_KNN(ctx):
    try: return float(skdim.id.KNN().fit(ctx.X3).dimension_) if ctx.X3 is not None else np.nan
    except: return np.nan
def E24_FisherS(ctx):
    try: return float(skdim.id.FisherS().fit(ctx.X3).dimension_) if ctx.X3 is not None else np.nan
    except: return np.nan
def E25_ESS(ctx):
    try: return float(skdim.id.ESS().fit(ctx.X3).dimension_) if ctx.X3 is not None else np.nan
    except: return np.nan
def E26_CD(ctx):
    try: return float(skdim.id.CD().fit(ctx.X3).dimension_) if ctx.X3 is not None else np.nan
    except: return np.nan
def E28_CorrInt(ctx):
    try: return float(skdim.id.CorrInt().fit(ctx.X3).dimension_) if ctx.X3 is not None else np.nan
    except: return np.nan

ALL_ESTIMATORS = {k: v for k, v in list(globals().items()) if k.startswith('E') and callable(v)}
if SKIP_DANKO:
    ALL_ESTIMATORS.pop('E20_DANCo', None)

print(f"  CPUs disponibles : {os.cpu_count()} (Backend: {BACKEND})")
print(f"  {len(ALL_ESTIMATORS)} estimateurs actifs{' (E20_DANCo retiré)' if SKIP_DANKO else ''}")

# ==============================================================================
# PHASE 1 : TÉLÉCHARGEMENT UNIQUE
# ==============================================================================
def download_and_read_zip(url, header_names=None):
    try:
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req, timeout=15) as resp:
            with zipfile.ZipFile(io.BytesIO(resp.read())) as z:
                with z.open(z.namelist()[0]) as f:
                    return pd.read_csv(f, names=header_names, header=0) if header_names else pd.read_csv(f)
    except: return None

def _fetch_oi_day(year, m, day):
    url = f"https://data.binance.vision/data/futures/um/daily/metrics/BTCUSDT/BTCUSDT-metrics-{year}-{m}-{day:02d}.zip"
    df_m = download_and_read_zip(url)
    if df_m is None or df_m.empty: return None
    df_m['timestamp'] = pd.to_datetime(df_m['create_time'], format='ISO8601')
    df_m.set_index('timestamp', inplace=True)
    return df_m[['sum_open_interest_value']]

_data_cache = {}

def load_data_transformed(mois):
    if mois in _data_cache: return _data_cache[mois]

    year, m = mois.split('-')
    cols = ['open_time','open','high','low','close','volume','close_time',
            'quote_volume','count','taker_buy_volume','taker_buy_quote_volume','ignore']
    df = download_and_read_zip(f"https://data.binance.vision/data/futures/um/monthly/klines/BTCUSDT/1h/BTCUSDT-1h-{year}-{m}.zip", header_names=cols)
    if df is None or df.empty:
        print(f"    ❌ Échec Klines pour {mois}"); return None

    df['timestamp'] = pd.to_datetime(df['open_time'], unit='ms')
    df.set_index('timestamp', inplace=True)

    with ThreadPoolExecutor(max_workers=16) as ex:
        futures = [ex.submit(_fetch_oi_day, year, m, day) for day in range(1, pd.Period(mois).days_in_month + 1)]
        oi_list = [f.result() for f in as_completed(futures) if f.result() is not None]

    if oi_list:
        df_oi = pd.concat(oi_list)
        df_oi = df_oi[~df_oi.index.duplicated(keep='last')].resample('1H').last()
        df = df.join(df_oi, how='left')
        df['sum_open_interest_value'] = df['sum_open_interest_value'].interpolate('linear').ffill().bfill()
        oi_val = df['sum_open_interest_value'].values
    else: oi_val = np.ones(len(df))

    df['buy_vol']  = df['taker_buy_volume']
    df['sell_vol'] = df['volume'] - df['taker_buy_volume']

    sig_prix  = np.diff(np.log(df['close'].values))
    sig_volat = np.log(df['high'].values[1:] / (df['low'].values[1:] + 1e-8))
    oi_s      = pd.Series(oi_val)
    sig_oi    = ((oi_s - oi_s.rolling(120, min_periods=30).mean()) / (oi_s.rolling(120, min_periods=30).std() + 1e-8)).bfill().values[1:]
    vol_ma20  = pd.Series(df['volume']).rolling(20, min_periods=1).mean().values
    sig_vol   = np.log1p(df['volume'].values[1:] / (vol_ma20[1:] + 1e-8))
    flow      = (df['buy_vol'].values[1:] - df['sell_vol'].values[1:]) / (df['volume'].values[1:] + 1e-8)
    sig_cvd   = pd.Series(flow).rolling(6, min_periods=1).mean().values

    signals = {'PRIX': sig_prix, 'OI': sig_oi, 'VOLUME': sig_vol, 'CVD': sig_cvd, 'VOLATILITE': sig_volat}
    print(f"    ✓ {mois} ingéré (Canaux: {len(signals)})")
    _data_cache[mois] = signals
    return signals

QUESTIONS = {
    "NIVEAU 1 (Régime Structuré vs Chaotique)": {"mois_A": ["2023-01", "2023-10", "2024-02"], "mois_B": ["2023-05", "2023-08", "2023-09"]},
    "NIVEAU 2 (Tension : Purge vs Compression)": {"mois_A": ["2022-05", "2022-11", "2024-04"], "mois_B": ["2023-05", "2023-07", "2024-06"]},
    "NIVEAU 3 (Cinétique : Explosion vs Apathie)": {"mois_A": ["2022-05", "2022-11", "2024-03"], "mois_B": ["2022-12", "2023-08", "2023-09"]},
    "NIVEAU 4 (Direction : UP vs DOWN)": {"mois_A": ["2023-01", "2023-10", "2024-02"], "mois_B": ["2023-08", "2024-04", "2024-08"]}
}
ALL_VARIABLES = ['PRIX', 'OI', 'VOLUME', 'CVD', 'VOLATILITE']

all_unique_months = sorted(set(m for q in QUESTIONS.values() for ms in [q['mois_A'], q['mois_B']] for m in ms))

print(f"\n{'═'*72}\n  PHASE 1 | Téléchargement {len(all_unique_months)} mois uniques (OI parallèle)\n{'═'*72}")
t_start = t1 = time.time()
for mois in all_unique_months: load_data_transformed(mois)
t_dl = time.time() - t1
print(f"  ✅ Téléchargements : {t_dl/60:.1f} min\n")

# ==============================================================================
# PHASE 2 : PRÉ-CALCUL ULTRA-FAST PARALLÈLE
# ==============================================================================
def _compute_signal_windows(mois, var_name, sig, funcs):
    # Slicing corrigé [0 : limite : STRIDE] -> Équivalence mathématique absolue avec le range d'origine
    limit = len(sig) - WINDOW
    windows = sliding_window_view(sig, WINDOW)[0:limit:STRIDE]

    res = {nom: [] for nom in funcs}
    for w in windows:
        ctx = WindowContext(w)
        for nom, func in funcs.items():
            try:
                v = func(ctx)
                if np.isfinite(v): res[nom].append(float(v))
            except: pass
    return mois, var_name, res

tasks = [(mois, var_name, _data_cache[mois][var_name]) for mois in all_unique_months for var_name in ALL_VARIABLES if mois in _data_cache]

print(f"{'═'*72}\n  PHASE 2 | Pré-calcul Multi-Process (Context Architect)\n          | {len(tasks)} signaux dispatchés aux CPUs\n{'═'*72}")
t2 = time.time()

raw_results = Parallel(n_jobs=N_JOBS, backend=BACKEND, verbose=5)(
    delayed(_compute_signal_windows)(mois, var_name, sig, ALL_ESTIMATORS)
    for mois, var_name, sig in tasks
)

PRECOMPUTED = {}
for mois, var_name, res in raw_results:
    for nom in ALL_ESTIMATORS:
        PRECOMPUTED[(mois, var_name, nom)] = res[nom]

t_pc = time.time() - t2
print(f"\n  ✅ Pré-calcul 100% mutualisé : {t_pc/60:.1f} min")

# ==============================================================================
# PHASE 3 : BOUCLE STATISTIQUE — Lookup O(1)
# ==============================================================================
MATRIX_SUMMARY = {}
ALL_TESTS      = []

for q_name, q_params in QUESTIONS.items():
    print(f"\n{'█'*90}\n  {q_name}\n{'█'*90}")

    for var_name in ALL_VARIABLES:
        print(f"\n   ➤ TEST SUR LA VARIABLE : {var_name}")
        results = []

        for nom in ALL_ESTIMATORS:
            vA = [v for m in q_params['mois_A'] for v in PRECOMPUTED.get((m, var_name, nom), [])]
            vB = [v for m in q_params['mois_B'] for v in PRECOMPUTED.get((m, var_name, nom), [])]

            if len(vA) > 10 and len(vB) > 10:
                stat, pval = mannwhitneyu(vA, vB, alternative='two-sided')
                n1, n2 = len(vA), len(vB)
                pooled_std = np.sqrt(((n1-1)*np.var(vA) + (n2-1)*np.var(vB)) / (n1+n2-2))
                cohen_d = abs((np.mean(vA) - np.mean(vB)) / pooled_std) if pooled_std > 0 else 0
                results.append({'nom': nom, 'cohen': cohen_d, 'pval': pval, 'muA': np.mean(vA), 'muB': np.mean(vB)})

        if results:
            pvals = [r['pval'] for r in results]
            try:
                _, pvals_corr, _, _ = multipletests(pvals, method='holm')
                for i, r in enumerate(results): r['pval_corr'] = pvals_corr[i]
            except:
                for r in results: r['pval_corr'] = r['pval']

        for r in results:
            ALL_TESTS.append({'q_name': q_name, 'var_name': var_name, **r})

        max_c = max((r['cohen'] for r in results), default=0)
        MATRIX_SUMMARY.setdefault(q_name, {})[var_name] = max_c

        results.sort(key=lambda x: x['cohen'], reverse=True)
        print(f"      {'Estimateur':<15} | {'Cohen-d':<7} | {'P-Val Brut':<10} | {'P-Val (L)':<10} | Verdict Local")
        print("      " + "─" * 75)
        for r in results[:5]:
            verdict = ("✅ FORT" if r['cohen'] > 0.5 and r.get('pval_corr', r['pval']) < 0.05 else ("⚠️ " if r['cohen'] > 0.3 else "❌ "))
            print(f"      {r['nom']:<15} | {r['cohen']:.3f}   | {r['pval']:.3e} | {r.get('pval_corr', r['pval']):.3e} | {verdict}")

# ==============================================================================
# 📊 SYNTHÈSE GLOBALE ET MATRICE
# ==============================================================================
print(f"\n\n{'═'*95}")
print(f"  📊 MATRICE COHEN-D MAXIMUM (La Diagonale de Vérité)")
print(f"{'═'*95}")
print(f"  {'Question':>30} | {'PRIX':>8} | {'OI':>8} | {'VOLUME':>8} | {'CVD':>8} | {'VOLATILITE':>10}")
print(f"  {'─'*30}-+-{'─'*8}-+-{'─'*8}-+-{'─'*8}-+-{'─'*8}-+-{'─'*10}")
for q_name, row in MATRIX_SUMMARY.items():
    short = q_name.split('(')[1].rstrip(')')[:30]
    print(f"  {short:>30} | {row.get('PRIX',0):.3f}   | {row.get('OI',0):.3f}   | "
          f"{row.get('VOLUME',0):.3f}   | {row.get('CVD',0):.3f}   | {row.get('VOLATILITE',0):.3f}")

pvals_global = [t['pval'] for t in ALL_TESTS]
try:
    _, pvals_gc, _, _ = multipletests(pvals_global, method='holm')
    for i, t in enumerate(ALL_TESTS): t['pval_global_corr'] = pvals_gc[i]
except:
    for t in ALL_TESTS: t['pval_global_corr'] = t['pval']

ALL_TESTS.sort(key=lambda x: x['cohen'], reverse=True)

print(f"\n{'═'*95}")
print(f"  🏆 TOP 15 GLOBAL (Holm sur {len(ALL_TESTS)} tests)")
print(f"{'═'*95}")
print(f"  {'Niveau':<10} | {'Variable':<10} | {'Estimateur':<15} | {'Cohen-d':<7} | {'P-Val Globale':<15} | Verdict")
print("  " + "─" * 93)
for t in ALL_TESTS[:15]:
    short_q = "Niveau " + t['q_name'].split()[1]
    verdict = "🔒 BLINDÉ" if t['cohen'] > 0.5 and t['pval_global_corr'] < 0.05 else "⚠️"
    print(f"  {short_q:<10} | {t['var_name']:<10} | {t['nom']:<15} | {t['cohen']:.3f}   | "
          f"{t['pval_global_corr']:.3e}     | {verdict}")

t_total = time.time() - t_start
print(f"\n  ⏱️  Durée totale        : {t_total/60:.1f} min")
print(f"  📡 Téléchargements     : {t_dl/60:.1f} min")
print(f"  🔢 Pré-calcul          : {t_pc/60:.1f} min")
print(f"{'═'*95}")

  CPUs disponibles : 2 (Backend: loky)
  27 estimateurs actifs

════════════════════════════════════════════════════════════════════════
  PHASE 1 | Téléchargement 14 mois uniques (OI parallèle)
════════════════════════════════════════════════════════════════════════
    ✓ 2022-05 ingéré (Canaux: 5)
    ✓ 2022-11 ingéré (Canaux: 5)
    ✓ 2022-12 ingéré (Canaux: 5)
    ✓ 2023-01 ingéré (Canaux: 5)
    ✓ 2023-05 ingéré (Canaux: 5)
    ✓ 2023-07 ingéré (Canaux: 5)
    ✓ 2023-08 ingéré (Canaux: 5)
    ✓ 2023-09 ingéré (Canaux: 5)
    ✓ 2023-10 ingéré (Canaux: 5)
    ✓ 2024-02 ingéré (Canaux: 5)
    ✓ 2024-03 ingéré (Canaux: 5)
    ✓ 2024-04 ingéré (Canaux: 5)
    ✓ 2024-06 ingéré (Canaux: 5)
    ✓ 2024-08 ingéré (Canaux: 5)
  ✅ Téléchargements : 0.3 min

════════════════════════════════════════════════════════════════════════
  PHASE 2 | Pré-calcul Multi-Process (Context Architect)
          | 70 signaux dispatchés aux CPUs
══════════════════════════════════════════════════════════════════

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=-1)]: Done  14 tasks      | elapsed: 120.8min
[Parallel(n_jobs=-1)]: Done  70 out of  70 | elapsed: 594.4min finished



  ✅ Pré-calcul 100% mutualisé : 594.4 min

██████████████████████████████████████████████████████████████████████████████████████████
  NIVEAU 1 (Régime Structuré vs Chaotique)
██████████████████████████████████████████████████████████████████████████████████████████

   ➤ TEST SUR LA VARIABLE : PRIX
      Estimateur      | Cohen-d | P-Val Brut | P-Val (L)  | Verdict Local
      ───────────────────────────────────────────────────────────────────────────
      E21_MOM         | 0.899   | 2.444e-14 | 6.355e-13 | ✅ FORT
      E16_MADA        | 0.706   | 3.700e-09 | 9.250e-08 | ✅ FORT
      E02_SampleEn    | 0.520   | 1.810e-06 | 4.345e-05 | ✅ FORT
      E13_CorrDim     | 0.464   | 4.474e-06 | 1.029e-04 | ⚠️ 
      E04_DFA         | 0.421   | 5.351e-04 | 1.124e-02 | ⚠️ 

   ➤ TEST SUR LA VARIABLE : OI
      Estimateur      | Cohen-d | P-Val Brut | P-Val (L)  | Verdict Local
      ───────────────────────────────────────────────────────────────────────────
      E22_TLE         | 0.840   | 